# VAE Medium Task — Enhanced Comparative Clustering

Enhancements over Easy Task:
- **MLP-VAE**, **Conv1D-VAE**, **Hybrid-VAE** (audio + TF-IDF lyrics)
- **3 Clustering algorithms**: K-Means · Agglomerative · DBSCAN
- **5 Metrics**: Silhouette · Davies-Bouldin · Calinski-H · ARI · NMI
- **3 Datasets**: FMA · LMD · GTZAN  — each with real Bangla audio (yt-dlp)
- Elbow plots, cluster heatmaps, language separation, VAE vs PCA Δ charts

## 0. Setup

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

In [ ]:
import warnings
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

from config.config import (
    BANGLA_DIR, BANGLA_QUERIES, BATCH_SIZE, BETA_DEFAULT,
    EPOCHS, FMA_METADATA_URL, GENRE_VOCAB, GTZAN_URLS,
    HIDDEN_DIMS, KMEANS_NINIT, LANG_COLORS, LANG_MARKERS,
    LATENT_DIM, LMD_URL, LR, LYRIC_DIM, MODEL_COLORS, N_BANGLA_PER_GENRE,
)
from src.data.bangla import get_bangla
from src.data.fma import download_fma_metadata, load_fma
from src.data.gtzan import download_gtzan_csv, load_gtzan
from src.data.lmd import download_lmd, load_lmd
from src.features.hybrid import make_hybrid
from src.models.conv_vae import ConvVAE
from src.models.mlp_vae import MLPVAE
from src.training.trainer import extract_latent, train_model
from src.clustering.engine import elbow_analysis, run_clustering
from src.visualization.plots import (
    plot_elbow, plot_metrics_heatmap, plot_training_curves,
    plot_cluster_composition, plot_language_separation,
)

warnings.filterwarnings("ignore")
np.random.seed(42); torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS = PROJECT_ROOT / "results" / "medium"
RESULTS.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)

## 1. Download Datasets

In [ ]:
FMA_DIR   = PROJECT_ROOT / "data" / "fma" / "fma_metadata"
LMD_DIR   = PROJECT_ROOT / "data" / "lmd"
GTZAN_CSV = PROJECT_ROOT / "data" / "gtzan" / "features_30_sec.csv"

download_fma_metadata(FMA_DIR, FMA_METADATA_URL)
download_lmd(LMD_DIR, LMD_URL)
download_gtzan_csv(GTZAN_CSV, GTZAN_URLS)

## 2. Load Datasets + Append Real Bangla

In [ ]:
X_fma, y_fma = load_fma(FMA_DIR)
X_lmd, y_lmd = load_lmd(LMD_DIR)
X_gh,  y_gh  = load_gtzan(GTZAN_CSV)

datasets_raw = []
for X_en, y_en, ds_name in [
    (X_fma, y_fma, "FMA (Free Music Archive)"),
    (X_lmd, y_lmd, "LMD (Lakh MIDI Dataset)"),
    (X_gh,  y_gh,  "GTZAN via GitHub"),
]:
    lang_en = np.array(["English"] * len(X_en))
    bX, by, bl = get_bangla(X_en.shape[1], BANGLA_QUERIES, BANGLA_DIR, N_BANGLA_PER_GENRE)
    if len(bX) > 0:
        X_m = np.vstack([X_en, bX]); y_m = np.concatenate([y_en, by]); l_m = np.concatenate([lang_en, bl])
    else:
        X_m, y_m, l_m = X_en, y_en, lang_en
    datasets_raw.append((X_m, y_m, l_m, ds_name))
    print(f"  {ds_name}: {X_m.shape}")

## 3. Run Full Pipeline (import from run_medium)

In [ ]:
from scripts.run_medium import full_pipeline

all_results = {}
for X_raw, y_labels, lang_labels, ds_name in datasets_raw:
    ds_key = ds_name.split()[0]
    all_results[ds_key] = full_pipeline(
        X_raw, y_labels, lang_labels, ds_name,
        latent_dim=LATENT_DIM, epochs=EPOCHS,
        device=DEVICE, results_dir=RESULTS,
    )
print("All experiments complete!")

## 4. Aggregate Metrics Table

In [ ]:
import pandas as pd
rows = []
label_map = {"mlp": "MLP-VAE", "conv": "Conv-VAE", "hybrid": "Hybrid-VAE", "pca": "PCA"}
for ds_key, res in all_results.items():
    for zkey, zlab in label_map.items():
        for algo in ["KMeans", "Agglomerative", "DBSCAN"]:
            r = res["cl"][zkey][algo]
            rows.append({"Dataset": ds_key, "Features": zlab, "Algorithm": algo,
                "Silhouette": r["sil"], "Davies-Bouldin": r["db"],
                "Calinski-H": r["ch"], "ARI": r["ari"], "NMI": r["nmi"]})
df_all = pd.DataFrame(rows)
df_all.to_csv(RESULTS / "full_metrics.csv", index=False)
print(df_all.to_string(index=False))

## 5. Metrics Heatmap

In [ ]:
plot_metrics_heatmap(df_all.rename(columns={"Features":"Model"}),
    save_path=RESULTS / "metrics_heatmap.png")
print(f"Results in: {RESULTS}")